In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_parquet('stock_data.parquet')
df.head(10)

,TradingDate,Symbol,OpenPrice,ClosePrice,HighPrice,LowPrice,Volume,Amount,ChangeRatio,TotalShare,...,F081001B,F081601B,F060101C,F060201C,F060301C,F060401C,PE,PB,PCF,PS
0,2010-01-01,000001,1185.217,1185.217,1185.217,1185.217,0.0,0.0,0.0,3105433762,...,0.114957,0.045584,-16.545267,NaN,-1.123885,-16.010482,NaN,NaN,NaN,NaN
1,2010-01-01,000415,73.966,73.966,73.966,73.966,0.0,0.0,0.0,300335834,...,NaN,3.280567,NaN,1.211856,-0.359215,NaN,NaN,NaN,NaN,NaN
2,2010-01-01,000661,87.656,87.656,87.656,87.656,0.0,0.0,0.0,131326570,...,-0.408958,-0.078347,3.566045,1.054451,0.411169,3.039289,NaN,NaN,NaN,NaN
3,2010-01-01,600153,82.029,82.029,82.029,82.029,0.0,0.0,0.0,1243194856,...,0.030121,0.009714,2.619866,1.172953,0.071868,2.113886,NaN,NaN,NaN,NaN
4,2010-01-01,000008,37.304,37.304,37.304,37.304,0.0,0.0,0.0,73653208,...,NaN,0.003900,10.222513,0.927288,0.332872,10.225908,NaN,NaN,NaN,NaN
5,2010-01-01,000009,48.041,48.041,48.041,48.041,0.0,0.0,0.0,1090750529,...,-0.366164,-0.394716,2.671310,1.302411,0.306810,2.279697,NaN,NaN,NaN,NaN
6,2010-01-01,000012,205.298,205.298,205.298,205.298,0.0,0.0,0.0,1223738124,...,0.553698,0.238748,2.716805,0.989766,0.308631,2.877369,NaN,NaN,NaN,NaN
7,2010-01-01,000021,114.930,114.930,114.930,114.930,0.0,0.0,0.0,879518521,...,0.235138,0.421827,1.360852,1.145383,0.031380,1.453359,NaN,NaN,NaN,NaN
8,2010-01-01,000024,122.483,122.483,122.483,122.483,0.0,0.0,0.0,1717300503,...,0.476394,0.174853,2.428407,1.840101,0.578410,2.006458,NaN,NaN,NaN,NaN
9,2010-01-01,000027,88.153,88.153,88.153,88.153,0.0,0.0,0.0,2202495332,...,-0.147415,0.474405,1.305207,1.127857,0.273453,2.304180,NaN,NaN,NaN,NaN


In [5]:
# 计算每一列的缺失率
nan_ratio = df.isnull().mean().sort_values(ascending=False)

# 打印缺失率超过 10% 的指标
print("--- 缺失率较高的指标 ---")
print(nan_ratio[nan_ratio > 0.1])

--- 缺失率较高的指标 ---
PCF         0.462982
PE          0.161784
F060401C    0.133681
F081001B    0.128016
F060101C    0.125173
F080701B    0.123706
F040604C    0.112874
F060201C    0.102714
F053301C    0.102269
dtype: float64


In [6]:
# 1. 设定严格的生死线：缺失率 > 35% 的直接删掉 (主要是 PCF)
high_missing_cols = nan_ratio[nan_ratio > 0.35].index.tolist()
df_refined = df.drop(columns=high_missing_cols)

# 2. 打印剩下的列中仍有缺失值的
remaining_nas = df_refined.isnull().mean()
print("--- 剩余待处理的缺失指标 ---")
print(remaining_nas[remaining_nas > 0])

--- 剩余待处理的缺失指标 ---
MarketValue              7.687760e-07
CirculatedMarketValue    7.687760e-07
F050204C                 2.392316e-02
F050504C                 2.657851e-02
F053301C                 1.022687e-01
F051401C                 2.235447e-02
F051501C                 2.235447e-02
F040304C                 6.674014e-02
F040604C                 1.128740e-01
F041405C                 2.466310e-02
F041705C                 2.453049e-02
F080701B                 1.237057e-01
F081001B                 1.280158e-01
F081601B                 1.593404e-02
F060101C                 1.251729e-01
F060201C                 1.027139e-01
F060301C                 2.243442e-02
F060401C                 1.336805e-01
PE                       1.617835e-01
PB                       9.777562e-02
PS                       9.617849e-02
dtype: float64


In [8]:
df_refined.to_parquet('hs300_daily_data.parquet', engine='pyarrow', index=False)

In [9]:
df = pd.read_parquet('hs300_daily_data.parquet')

In [10]:
import pandas as pd
import numpy as np

def comprehensive_missing_report(df):
    # 1. 基础统计：每一列的缺失总数和比例
    total_rows = len(df)
    report = pd.DataFrame({
        'Missing_Count': df.isnull().sum(),
        'Missing_Rate': (df.isnull().sum() / total_rows) * 100
    })
    
    # 2. 计算每一列的有效数据量 (Non-null)
    report['Non_Null_Count'] = df.notnull().sum()
    
    # 3. 按年份拆解缺失率 (识别因子是否是近几年才有的)
    df['Year'] = pd.to_datetime(df['TradingDate']).dt.year
    yearly_report = df.groupby('Year').apply(lambda x: x.isnull().mean() * 100)
    
    # 找出缺失最严重的年份和对应的缺失率
    report['Max_Yearly_Missing'] = yearly_report.max()
    report['Min_Yearly_Missing'] = yearly_report.min()
    
    # 排序：按缺失率从高到低
    report = report.sort_values(by='Missing_Rate', ascending=False)
    
    return report, yearly_report

In [11]:
# 执行检查
missing_stats, yearly_detail = comprehensive_missing_report(df)

print("--- 因子缺失值体检报告 (Top 15) ---")
missing_stats.head(15).round(2)

--- 因子缺失值体检报告 (Top 15) ---


C:\Users\HP\AppData\Local\Temp\ipykernel_27516\3286997321.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  yearly_report = df.groupby('Year').apply(lambda x: x.isnull().mean() * 100)


,Missing_Count,Missing_Rate,Non_Null_Count,Max_Yearly_Missing,Min_Yearly_Missing
PE,420886,16.18,2180652,21.65,11.32
F060401C,347775,13.37,2253763,19.60,7.79
F081001B,333038,12.80,2268500,16.50,9.31
F060101C,325642,12.52,2275896,17.13,7.50
F080701B,321825,12.37,2279713,15.56,8.80
F040604C,293646,11.29,2307892,14.42,8.45
F060201C,267214,10.27,2334324,13.12,7.45
F053301C,266056,10.23,2335482,13.02,7.39
PB,254367,9.78,2347171,15.67,7.09
PS,250212,9.62,2351326,15.67,7.01


In [12]:
df

,TradingDate,Symbol,OpenPrice,ClosePrice,HighPrice,LowPrice,Volume,Amount,ChangeRatio,TotalShare,...,F081001B,F081601B,F060101C,F060201C,F060301C,F060401C,PE,PB,PS,Year
0,2010-01-01,000001,1185.217,1185.217,1185.217,1185.217,0.0,0.000000e+00,0.00000,3105433762,...,0.114957,0.045584,-16.545267,NaN,-1.123885,-16.010482,NaN,NaN,NaN,2010
1,2010-01-01,000415,73.966,73.966,73.966,73.966,0.0,0.000000e+00,0.00000,300335834,...,NaN,3.280567,NaN,1.211856,-0.359215,NaN,NaN,NaN,NaN,2010
2,2010-01-01,000661,87.656,87.656,87.656,87.656,0.0,0.000000e+00,0.00000,131326570,...,-0.408958,-0.078347,3.566045,1.054451,0.411169,3.039289,NaN,NaN,NaN,2010
3,2010-01-01,600153,82.029,82.029,82.029,82.029,0.0,0.000000e+00,0.00000,1243194856,...,0.030121,0.009714,2.619866,1.172953,0.071868,2.113886,NaN,NaN,NaN,2010
4,2010-01-01,000008,37.304,37.304,37.304,37.304,0.0,0.000000e+00,0.00000,73653208,...,NaN,0.003900,10.222513,0.927288,0.332872,10.225908,NaN,NaN,NaN,2010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2601533,2024-12-31,600196,370.206,364.053,371.085,363.614,12275863.0,3.070969e+08,-0.01701,2671326465,...,0.268856,0.013926,1.427370,0.991132,0.094667,1.281921,27.818553,1.172498,1.603459,2024
2601534,2024-12-31,601377,17.859,17.173,17.887,17.173,76172078.0,4.846399e+08,-0.03841,8635987294,...,-0.279491,0.099453,8.346555,NaN,2.300396,8.522364,27.520911,0.886677,5.537064,2024
2601535,2024-12-31,688009,7.381,7.370,7.511,7.346,47040674.0,2.969310e+08,0.00000,10589819000,...,-0.247170,-0.152055,1.653276,1.135159,0.187732,1.419449,19.064147,1.356368,1.787460,2024
2601536,2024-12-31,688256,684.000,658.000,688.000,658.000,7021055.0,4.694181e+09,-0.03731,417456753,...,NaN,2.082956,NaN,0.678117,-2.511924,NaN,NaN,47.942255,387.216998,2024


In [13]:
import pandas as pd
import numpy as np
import time

def professional_clean_pipeline_with_stats(df):
    start_time = time.time()
    print(f"开始数据清洗流程，原始行数: {len(df)}")

    # --- 1. 基础脱水 ---
    df = df.dropna(subset=['ClosePrice']).copy()
    
    # --- 2. 重构 AvgPrice ---
    df['AvgPrice'] = df['Amount'] / df['Volume'].replace(0, np.nan)
    df['AvgPrice'] = df['AvgPrice'].fillna(df['ClosePrice'])
    
    # --- 3. 定义特征列 ---
    valuation_cols = ['PE', 'PB', 'PS']
    f_cols = [col for col in df.columns if col.startswith('F')]
    all_features = valuation_cols + f_cols

    # --- 4. 排序 ---
    df = df.sort_values(['Symbol', 'TradingDate'])

    # 用于存储统计结果
    stats_list = []

    # --- 5. 核心填充逻辑 ---
    for col in all_features:
        col_start = time.time()
        
        # 记录初始缺失数
        total_missing = df[col].isnull().sum()
        
        if total_missing == 0:
            stats_list.append({
                'Factor': col, 'Initial_Missing': 0, 
                'ffill%': 0, 'Sector%': 0, 'Daily%': 0, 'Zero%': 0
            })
            print(f"因子 [{col}] 无缺失，跳过。")
            continue

        # A. 时序前向填充 (ffill)
        df[col] = df.groupby('Symbol')[col].ffill()
        after_ffill = df[col].isnull().sum()
        count_ffill = total_missing - after_ffill
        
        # B. 行业中位数填充
        sector_med = df.groupby(['TradingDate', 'Indexcode'])[col].transform('median')
        df[col] = df[col].fillna(sector_med)
        after_sector = df[col].isnull().sum()
        count_sector = after_ffill - after_sector
        
        # C. 全市场每日中位数补漏
        daily_med = df.groupby('TradingDate')[col].transform('median')
        df[col] = df[col].fillna(daily_med)
        after_daily = df[col].isnull().sum()
        count_daily = after_sector - after_daily
        
        # D. 终极补 0
        df[col] = df[col].fillna(0)
        count_zero = after_daily
        
        # 计算比例
        stats_list.append({
            'Factor': col,
            'Initial_Missing': total_missing,
            'ffill%': round(count_ffill / total_missing * 100, 2),
            'Sector%': round(count_sector / total_missing * 100, 2),
            'Daily%': round(count_daily / total_missing * 100, 2),
            'Zero%': round(count_zero / total_missing * 100, 2)
        })
        
        print(f"因子 [{col}] 完成 | ffill: {count_ffill} | Sector: {count_sector} | 耗时: {time.time() - col_start:.2f}s")

    # --- 6. 整理统计报表 ---
    stats_df = pd.DataFrame(stats_list).set_index('Factor')
    

    print(f"\n--- 全流程清洗完毕 ---")
    print(f"总耗时: {(time.time() - start_time) / 60:.2f} 分钟")
    
    return df, stats_df

# 执行清洗并获取报表
df_clean, clean_stats = professional_clean_pipeline_with_stats(df)

# 查看报表
print("\n--- 填充比例统计表 ---")
print(clean_stats)

开始数据清洗流程，原始行数: 2601538
因子 [PE] 完成 | ffill: 406606 | Sector: 13332 | 耗时: 0.64s
因子 [PB] 完成 | ffill: 249881 | Sector: 3814 | 耗时: 0.57s
因子 [PS] 完成 | ffill: 247085 | Sector: 2455 | 耗时: 0.56s
因子 [F050204C] 完成 | ffill: 27680 | Sector: 32349 | 耗时: 0.53s
因子 [F050504C] 完成 | ffill: 32201 | Sector: 34222 | 耗时: 0.53s
因子 [F053301C] 完成 | ffill: 43093 | Sector: 124673 | 耗时: 0.51s
因子 [F051401C] 完成 | ffill: 35374 | Sector: 20996 | 耗时: 0.52s
因子 [F051501C] 完成 | ffill: 35374 | Sector: 20996 | 耗时: 0.51s
因子 [F040304C] 完成 | ffill: 46674 | Sector: 125614 | 耗时: 0.51s
因子 [F040604C] 完成 | ffill: 54692 | Sector: 238954 | 耗时: 0.51s
因子 [F041405C] 完成 | ffill: 27483 | Sector: 35010 | 耗时: 0.52s
因子 [F041705C] 完成 | ffill: 27138 | Sector: 35010 | 耗时: 0.52s
因子 [F080701B] 完成 | ffill: 275529 | Sector: 43462 | 耗时: 0.51s
因子 [F081001B] 完成 | ffill: 285663 | Sector: 45055 | 耗时: 0.51s
因子 [F081601B] 完成 | ffill: 3722 | Sector: 37731 | 耗时: 0.51s
因子 [F060101C] 完成 | ffill: 288343 | Sector: 35166 | 耗时: 0.52s
因子 [F060201C] 完成 | ffill: 426

In [14]:
df_clean

,TradingDate,Symbol,OpenPrice,ClosePrice,HighPrice,LowPrice,Volume,Amount,ChangeRatio,TotalShare,...,F081601B,F060101C,F060201C,F060301C,F060401C,PE,PB,PS,Year,AvgPrice
0,2010-01-01,000001,1185.217,1185.217,1185.217,1185.217,0.0,0.000000e+00,0.00000,3105433762,...,0.0,-16.545267,1.041037,-1.123885,-16.010482,0.000000,0.000000,0.000000,2010,1185.217000
664,2010-01-04,000001,1192.512,1153.118,1195.430,1151.659,24192276.0,5.802495e+08,-0.02708,3105433762,...,0.0,-16.545267,1.041037,-1.123885,-16.010482,14.636017,3.597032,5.151150,2010,23.984906
1360,2010-01-05,000001,1155.064,1133.178,1162.359,1106.429,55649982.0,1.293477e+09,-0.01729,3105433762,...,0.0,-16.545267,1.041037,-1.123885,-16.010482,14.382927,3.534831,5.062075,2010,23.243079
1449,2010-01-06,000001,1130.747,1113.725,1130.747,1104.970,41214313.0,9.444537e+08,-0.01717,3105433762,...,0.0,-16.545267,1.041037,-1.123885,-16.010482,14.136010,3.474147,4.975172,2010,22.915672
1937,2010-01-07,000001,1113.725,1101.566,1121.020,1089.407,35533685.0,8.041663e+08,-0.01092,3105433762,...,0.0,-16.545267,1.040979,-1.123885,-16.010482,13.981686,3.436220,4.920858,2010,22.631098
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2598544,2024-12-25,688981,96.480,97.990,99.800,96.380,98532907.0,9.679610e+09,0.01167,7975539838,...,0.0,3.928365,0.926195,0.350992,3.583014,162.047126,3.577252,17.271068,2024,98.237335
2599235,2024-12-26,688981,98.000,96.730,98.880,96.150,71104586.0,6.914605e+09,-0.01286,7975539838,...,0.0,3.928365,0.926195,0.350992,3.583014,159.963450,3.531254,17.048988,2024,97.245563
2600032,2024-12-27,688981,96.780,97.510,102.370,96.490,114471630.0,1.137894e+10,0.00806,7975539838,...,0.0,3.928365,0.926195,0.350992,3.583014,161.253345,3.559729,17.186466,2024,99.404053
2600669,2024-12-30,688981,96.600,99.290,100.530,96.000,90657304.0,8.950495e+09,0.01825,7975539838,...,0.0,3.928365,0.926195,0.350992,3.583014,164.196950,3.624710,17.500197,2024,98.728886


In [15]:
def final_check_report(df_clean):
    # 1. 统计特征列
    valuation_cols = ['PE', 'PB', 'PS', 'AvgPrice']
    f_cols = [col for col in df_clean.columns if col.startswith('F')]
    check_cols = valuation_cols + f_cols
    
    # 2. 计算缺失值和零值占比
    total_rows = len(df_clean)
    check_stats = []
    
    for col in check_cols:
        if col in df_clean.columns:
            nan_count = df_clean[col].isnull().sum()
            zero_count = (df_clean[col] == 0).sum()
            check_stats.append({
                'Factor': col,
                'NaN_Count': nan_count,
                'Zero_Count': zero_count,
                'Zero_Rate(%)': round((zero_count / total_rows) * 100, 2),
                'Mean': round(df_clean[col].mean(), 4),
                'Std': round(df_clean[col].std(), 4)
            })
    
    report_df = pd.DataFrame(check_stats).set_index('Factor')
    return report_df

# 执行复核
final_report = final_check_report(df_clean)

print("--- 清洗后因子质量终审报告 ---")
print(final_report)

# 检查是否有任何 NaN 遗留
remaining_total_nan = df_clean.isnull().sum().sum()
print(f"\n全表剩余 NaN 总数: {remaining_total_nan}")

--- 清洗后因子质量终审报告 ---
          NaN_Count  Zero_Count  Zero_Rate(%)       Mean          Std
Factor                                                               
PE                0         483          0.02    82.7279     358.7258
PB                0         483          0.02     4.1422      11.3025
PS                0         483          0.02     7.9164     153.8234
AvgPrice          0           0          0.00    44.9802     543.9315
F050204C          0           0          0.00     0.0518       0.0849
F050504C          0           0          0.00     0.0697       0.9490
F053301C          0           0          0.00     1.3468     183.9316
F051401C          0           0          0.00 -1852.0569  321932.4856
F051501C          0          87          0.00 -1209.3240  210137.4606
F040304C          0           0          0.00   180.2671   11129.0687
F040604C          0     2601538        100.00     0.0000       0.0000
F041405C          0           0          0.00    14.4809     133.9859


In [16]:
df_sorted2 = df_clean.sort_values(by=['TradingDate', 'Symbol'])

# 3. 重置索引（可选，但推荐，防止索引混乱）
df_sorted2 = df_sorted2.reset_index(drop=True)

In [17]:
df_sorted2.to_parquet('hs300_daily_data_final.parquet', engine='pyarrow', index=False)

In [18]:
df = pd.read_parquet('hs300_daily_data_final.parquet')

In [19]:
df

,TradingDate,Symbol,OpenPrice,ClosePrice,HighPrice,LowPrice,Volume,Amount,ChangeRatio,TotalShare,...,F081601B,F060101C,F060201C,F060301C,F060401C,PE,PB,PS,Year,AvgPrice
0,2010-01-01,000001,1185.217,1185.217,1185.217,1185.217,0.0,0.000000e+00,0.00000,3105433762,...,0.0,-16.545267,1.041037,-1.123885,-16.010482,0.000000,0.000000,0.000000,2010,1185.217000
1,2010-01-01,000002,940.503,940.503,940.503,940.503,0.0,0.000000e+00,0.00000,10995210218,...,0.0,2.325104,1.086545,0.274957,1.729696,0.000000,0.000000,0.000000,2010,940.503000
2,2010-01-01,000008,37.304,37.304,37.304,37.304,0.0,0.000000e+00,0.00000,73653208,...,0.0,10.222513,0.927288,0.332872,10.225908,0.000000,0.000000,0.000000,2010,37.304000
3,2010-01-01,000009,48.041,48.041,48.041,48.041,0.0,0.000000e+00,0.00000,1090750529,...,0.0,2.671310,1.302411,0.306810,2.279697,0.000000,0.000000,0.000000,2010,48.041000
4,2010-01-01,000012,205.298,205.298,205.298,205.298,0.0,0.000000e+00,0.00000,1223738124,...,0.0,2.716805,0.989766,0.308631,2.877369,0.000000,0.000000,0.000000,2010,205.298000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2601533,2024-12-31,688472,12.308,12.723,13.047,12.267,44994177.0,5.635011e+08,0.03120,3688217324,...,0.0,2.582219,1.459607,0.110176,2.094439,15.955231,2.156634,0.902834,2024,12.523868
2601534,2024-12-31,688506,192.970,191.730,197.400,191.010,750140.0,1.454740e+08,-0.00509,401000000,...,0.0,1.147169,1.053099,0.745581,1.059434,27.066889,506.235987,136.835264,2024,193.929183
2601535,2024-12-31,688561,28.560,26.830,28.690,26.830,6358241.0,1.744505e+08,-0.05060,685172377,...,0.0,-2.623722,0.932970,-0.058607,-4.523994,256.209924,1.805473,2.853428,2024,27.436913
2601536,2024-12-31,688599,21.392,20.561,21.413,20.561,18658830.0,3.669588e+08,-0.03596,2179365328,...,0.0,7.570764,0.981195,0.184174,7.330657,7.604313,1.150976,0.370942,2024,19.666765


In [22]:
# 1. 确保日期格式正确
df['TradingDate'] = pd.to_datetime(df['TradingDate'])

# 2. 核心操作：删除成交量为 0 的行
# 注意：有些极低价股成交额极小但不为0，所以我们严格按 Volume == 0 过滤
print(f"清洗前行数: {len(df)}")
df = df[df['Volume'] > 0].copy()
print(f"清洗后行数: {len(df)}")

# 3. 排序（这是为了下一步计算收益率做准备）
# 必须先按代码、再按日期排序，确保每只股票的数据是按时间顺序排列的
df = df.sort_values(['Symbol', 'TradingDate']).reset_index(drop=True)

# 4. 检查是否有重复行（可选，但建议执行）
df = df.drop_duplicates(subset=['TradingDate', 'Symbol'])

print("✅ 已成功剔除成交量为0的非交易日/停牌数据。")

清洗前行数: 2601538
清洗后行数: 2351998
✅ 已成功剔除成交量为0的非交易日/停牌数据。


In [23]:
df.head(10)

,TradingDate,Symbol,OpenPrice,ClosePrice,HighPrice,LowPrice,Volume,Amount,ChangeRatio,TotalShare,...,F081601B,F060101C,F060201C,F060301C,F060401C,PE,PB,PS,Year,AvgPrice
0,2010-01-04,000001,1192.512,1153.118,1195.430,1151.659,24192276.0,5.802495e+08,-0.02708,3105433762,...,0.0,-16.545267,1.041037,-1.123885,-16.010482,14.636017,3.597032,5.151150,2010,23.984906
1,2010-01-05,000001,1155.064,1133.178,1162.359,1106.429,55649982.0,1.293477e+09,-0.01729,3105433762,...,0.0,-16.545267,1.041037,-1.123885,-16.010482,14.382927,3.534831,5.062075,2010,23.243079
2,2010-01-06,000001,1130.747,1113.725,1130.747,1104.970,41214313.0,9.444537e+08,-0.01717,3105433762,...,0.0,-16.545267,1.041037,-1.123885,-16.010482,14.136010,3.474147,4.975172,2010,22.915672
3,2010-01-07,000001,1113.725,1101.566,1121.020,1089.407,35533685.0,8.041663e+08,-0.01092,3105433762,...,0.0,-16.545267,1.040979,-1.123885,-16.010482,13.981686,3.436220,4.920858,2010,22.631098
4,2010-01-08,000001,1094.271,1099.134,1106.429,1086.976,28854306.0,6.506674e+08,-0.00221,3105433762,...,0.0,-16.545267,1.040979,-1.123885,-16.010482,13.950822,3.428634,4.909996,2010,22.550097
5,2010-01-11,000001,1142.905,1099.134,1151.659,1083.571,44284602.0,1.009986e+09,0.00000,3105433762,...,0.0,-16.545267,1.040979,-1.123885,-16.010482,13.950822,3.428634,4.909996,2010,22.806699
6,2010-01-12,000001,1097.675,1091.839,1101.566,1060.227,59179591.0,1.310069e+09,-0.00664,3105433762,...,0.0,-16.545267,1.040979,-1.123885,-16.010482,13.858228,3.405878,4.877407,2010,22.137176
7,2010-01-13,000001,1063.631,1019.374,1065.090,1011.593,93503947.0,1.990493e+09,-0.06637,3105433762,...,0.0,-16.545267,1.040979,-1.123885,-16.010482,12.938461,3.179831,4.553695,2010,21.287794
8,2010-01-14,000001,1021.319,1019.860,1029.101,1001.866,52119474.0,1.089329e+09,0.00048,3105433762,...,0.0,-16.545267,1.040979,-1.123885,-16.010482,12.944634,3.181348,4.555868,2010,20.900623
9,2010-01-15,000001,1022.292,1042.232,1052.932,1004.297,53950846.0,1.146375e+09,0.02194,3105433762,...,0.0,-16.545267,1.040979,-1.123885,-16.010482,13.228589,3.251134,4.655805,2010,21.248514


In [24]:
import numpy as np

# 1. 确保时序正确 (按股票、日期排序)
df = df.sort_values(['Symbol', 'TradingDate'])

# 2. 分别计算简单收益率和对数收益率
# 使用 groupby 确保收益率不会跨股票计算
df['Simple_Ret'] = df.groupby('Symbol')['ClosePrice'].pct_change()
df['Log_Ret'] = df.groupby('Symbol')['ClosePrice'].transform(lambda x: np.log(x) - np.log(x.shift(1)))

# 3. 删除 2010-01-04 的数据
# 因为这是全样本的起始日，所有股票的收益率在这一天都是 NaN
df = df[df['TradingDate'] > '2010-01-04'].reset_index(drop=True)

# 4. 重新排序：先按日期，再按股票代码
# 这种排序符合“截面”逻辑，方便后续按天处理数据
df = df.sort_values(['TradingDate', 'Symbol']).reset_index(drop=True)

# 验证结果
print(f"清洗后剩余行数: {len(df)}")
print(df[['TradingDate', 'Symbol', 'Simple_Ret', 'Log_Ret']].head())

清洗后剩余行数: 2351529
  TradingDate  Symbol  Simple_Ret   Log_Ret
0  2010-01-05  000001   -0.017292 -0.017444
1  2010-01-05  000002   -0.022641 -0.022901
2  2010-01-05  000009   -0.025828 -0.026167
3  2010-01-05  000012   -0.016999 -0.017145
4  2010-01-05  000021    0.022902  0.022644


In [25]:
df.head(10)

,TradingDate,Symbol,OpenPrice,ClosePrice,HighPrice,LowPrice,Volume,Amount,ChangeRatio,TotalShare,...,F060201C,F060301C,F060401C,PE,PB,PS,Year,AvgPrice,Simple_Ret,Log_Ret
0,2010-01-05,000001,1155.064,1133.178,1162.359,1106.429,55649982.0,1.293477e+09,-0.01729,3105433762,...,1.041037,-1.123885,-16.010482,14.382927,3.534831,5.062075,2010,23.243079,-0.017292,-0.017444
1,2010-01-05,000002,914.402,901.352,915.272,887.431,184862078.0,1.910014e+09,-0.02264,10995210218,...,1.086545,0.274957,1.729696,21.372605,2.508569,2.330360,2010,10.332105,-0.022641,-0.022901
2,2010-01-05,000009,47.647,46.204,47.910,45.766,31463320.0,3.320040e+08,-0.02583,1090750529,...,1.302411,0.306810,2.279697,45.444410,3.858595,3.523082,2010,10.552095,-0.025828,-0.026167
3,2010-01-05,000012,202.470,199.852,204.251,197.966,10199239.0,1.944016e+08,-0.01700,1223738124,...,0.989766,0.308631,2.877369,28.065486,4.161223,4.422898,2010,19.060405,-0.016999,-0.017145
4,2010-01-05,000021,116.446,119.477,122.152,115.465,20586249.0,2.766909e+08,0.02290,879518521,...,1.145383,0.031380,1.453359,45.570924,2.981567,0.879170,2010,13.440569,0.022902,0.022644
5,2010-01-05,000024,118.114,113.101,118.114,111.767,49727812.0,1.228104e+09,-0.04505,1717300503,...,1.840101,0.578410,2.006458,25.684139,2.307307,4.165483,2010,24.696527,-0.045045,-0.046092
6,2010-01-05,000027,87.178,87.113,87.698,85.878,9238349.0,1.234909e+08,-0.00075,2202495332,...,1.127857,0.273453,2.304180,14.791031,1.881693,2.591473,2010,13.367206,-0.000746,-0.000746
7,2010-01-05,000031,146.437,143.891,146.437,140.676,25390266.0,2.715443e+08,-0.02096,1813731596,...,1.462186,-0.012874,-0.052093,52.151554,3.013993,9.674246,2010,10.694819,-0.020963,-0.021186
8,2010-01-05,000039,272.818,276.569,279.279,266.357,18257914.0,2.393941e+08,0.01375,2662396051,...,1.840245,0.273920,5.302863,36.841722,2.232313,1.725476,2010,13.111796,0.013749,0.013655
9,2010-01-05,000046,193.476,185.959,193.476,181.784,10136227.0,1.354171e+08,-0.03468,2263695884,...,1.305871,-0.098249,-0.419185,75.073801,3.342431,12.712486,2010,13.359712,-0.034681,-0.035297


In [26]:
df.to_parquet('hs300_data.parquet', engine='pyarrow', index=False)

In [ ]:
import pandas as pd
df = pd.read_parquet('hs300_data.parquet')

,TradingDate,Symbol,OpenPrice,ClosePrice,HighPrice,LowPrice,Volume,Amount,ChangeRatio,TotalShare,...,F060201C,F060301C,F060401C,PE,PB,PS,Year,AvgPrice,Simple_Ret,Log_Ret
0,2010-01-05,000001,1155.064,1133.178,1162.359,1106.429,55649982.0,1.293477e+09,-0.01729,3105433762,...,1.041037,-1.123885,-16.010482,14.382927,3.534831,5.062075,2010,23.243079,-0.017292,-0.017444
1,2010-01-05,000002,914.402,901.352,915.272,887.431,184862078.0,1.910014e+09,-0.02264,10995210218,...,1.086545,0.274957,1.729696,21.372605,2.508569,2.330360,2010,10.332105,-0.022641,-0.022901
2,2010-01-05,000009,47.647,46.204,47.910,45.766,31463320.0,3.320040e+08,-0.02583,1090750529,...,1.302411,0.306810,2.279697,45.444410,3.858595,3.523082,2010,10.552095,-0.025828,-0.026167
3,2010-01-05,000012,202.470,199.852,204.251,197.966,10199239.0,1.944016e+08,-0.01700,1223738124,...,0.989766,0.308631,2.877369,28.065486,4.161223,4.422898,2010,19.060405,-0.016999,-0.017145
4,2010-01-05,000021,116.446,119.477,122.152,115.465,20586249.0,2.766909e+08,0.02290,879518521,...,1.145383,0.031380,1.453359,45.570924,2.981567,0.879170,2010,13.440569,0.022902,0.022644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2351524,2024-12-31,688472,12.308,12.723,13.047,12.267,44994177.0,5.635011e+08,0.03120,3688217324,...,1.459607,0.110176,2.094439,15.955231,2.156634,0.902834,2024,12.523868,0.031204,0.030727
2351525,2024-12-31,688506,192.970,191.730,197.400,191.010,750140.0,1.454740e+08,-0.00509,401000000,...,1.053099,0.745581,1.059434,27.066889,506.235987,136.835264,2024,193.929183,-0.005085,-0.005098
2351526,2024-12-31,688561,28.560,26.830,28.690,26.830,6358241.0,1.744505e+08,-0.05060,685172377,...,0.932970,-0.058607,-4.523994,256.209924,1.805473,2.853428,2024,27.436913,-0.050602,-0.051927
2351527,2024-12-31,688599,21.392,20.561,21.413,20.561,18658830.0,3.669588e+08,-0.03596,2179365328,...,0.981195,0.184174,7.330657,7.604313,1.150976,0.370942,2024,19.666765,-0.035962,-0.036625


In [7]:
df_sub = df[df['Symbol'] == '688981']
df_sub

,TradingDate,Symbol,OpenPrice,ClosePrice,HighPrice,LowPrice,Volume,Amount,ChangeRatio,TotalShare,...,F060201C,F060301C,F060401C,PE,PB,PS,Year,AvgPrice,Simple_Ret,Log_Ret
1541212,2020-07-16,688981,95.00,82.92,95.00,80.00,552103902.0,4.796717e+10,2.01966,7415475343,...,1.112617,0.380912,5.855445,342.793788,8.628959,27.926900,2020,86.880696,NaN,NaN
1541940,2020-07-17,688981,79.00,77.06,84.90,75.00,219471588.0,1.738815e+10,-0.07067,7415475343,...,1.112617,0.380912,5.855445,318.568371,8.019146,25.953291,2020,79.227330,-0.070671,-0.073292
1542669,2020-07-20,688981,77.19,79.17,80.51,70.02,228583338.0,1.700523e+10,0.02738,7424033430,...,1.112617,0.380912,5.855445,327.668897,8.248229,26.694698,2020,74.393998,0.027381,0.027013
1543398,2020-07-21,688981,78.30,78.63,82.89,77.77,161873271.0,1.297766e+10,-0.00682,7424033430,...,1.112617,0.380912,5.855445,325.433944,8.191969,26.512619,2020,80.171753,-0.006821,-0.006844
1544127,2020-07-22,688981,77.80,79.57,81.78,77.20,133942790.0,1.068249e+10,0.01195,7424033430,...,1.112617,0.380912,5.855445,329.324417,8.289902,26.829570,2020,79.754156,0.011955,0.011884
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2348544,2024-12-25,688981,96.48,97.99,99.80,96.38,98532907.0,9.679610e+09,0.01167,7975539838,...,0.926195,0.350992,3.583014,162.047126,3.577252,17.271068,2024,98.237335,0.011666,0.011599
2349290,2024-12-26,688981,98.00,96.73,98.88,96.15,71104586.0,6.914605e+09,-0.01286,7975539838,...,0.926195,0.350992,3.583014,159.963450,3.531254,17.048988,2024,97.245563,-0.012858,-0.012942
2350035,2024-12-27,688981,96.78,97.51,102.37,96.49,114471630.0,1.137894e+10,0.00806,7975539838,...,0.926195,0.350992,3.583014,161.253345,3.559729,17.186466,2024,99.404053,0.008064,0.008031
2350781,2024-12-30,688981,96.60,99.29,100.53,96.00,90657304.0,8.950495e+09,0.01825,7975539838,...,0.926195,0.350992,3.583014,164.196950,3.624710,17.500197,2024,98.728886,0.018255,0.018090
